# EduNexus AI Lab Instructor — 4-Bit QLoRA Fine-Tuning

This notebook trains a specialized **Qwen2.5-3B / Qwen3** model on EduNexus electronic circuit diagnostics, wiring rules, fault detection, and next-step teaching recommendations.

### Features:
- **4-bit NormalFloat (NF4)** quantization with `bitsandbytes`
- **LoRA (Rank=8, Alpha=16)** for parameter-efficient adaptation (<0.5% weights updated)
- **Fast training**: ~3–5 minutes on a free Google Colab T4 GPU (or locally on NVIDIA RTX 2050)
- **Direct Export**: Produces LoRA adapter and instructions for Ollama deployment

In [ ]:
# Step 1: Install fine-tuning dependencies
!pip install -q --upgrade torch torchvision
!pip install -q transformers peft bitsandbytes datasets trl accelerate

In [ ]:
# Step 2: Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 3: Load or Upload Dataset
# If running in Colab, upload edunexus_train.jsonl and edunexus_eval.jsonl
import json
from pathlib import Path
from datasets import load_dataset

train_file = "edunexus_train.jsonl"
eval_file = "edunexus_eval.jsonl"

if not Path(train_file).exists():
    print(f"Please upload '{train_file}' to the current directory.")
else:
    dataset = load_dataset("json", data_files={"train": train_file, "eval": eval_file})
    print(f"Train samples: {len(dataset['train'])}, Eval samples: {len(dataset['eval'])}")
    print("Sample prompt:", dataset['train'][0]['messages'][1]['content'][:200])

In [ ]:
# Step 4: Configure 4-bit Quantization and LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./edunexus_qwen3_lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Train Adapter
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    peft_config=lora_config,
    dataset_text_field="messages",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training complete! Adapter saved to {OUTPUT_DIR}")

In [ ]:
# Step 6: Download the Adapter (for Colab users)
import shutil
shutil.make_archive("edunexus_qwen3_lora", "zip", OUTPUT_DIR)
print("Downloaded 'edunexus_qwen3_lora.zip'. Extract into your EduNexus 'models/' directory!")